# ALS implicit best config (PyTorch 2 GPU) + full test evaluation

In [ ]:

import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:

base = Path('/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output')
files = sorted(base.glob('test/*.parquet'))
dfs = [pq.read_table(f).to_pandas() for f in files]
inter = pd.concat(dfs, ignore_index=True)
users_raw = inter['user_idx'].to_numpy()
items_raw = inter['item_idx'].to_numpy()
ratings = inter['rating'].fillna(1.0).to_numpy(dtype=np.float32)
u_uniq, u_inv = np.unique(users_raw, return_inverse=True)
i_uniq, i_inv = np.unique(items_raw, return_inverse=True)
U = len(u_uniq); I = len(i_uniq)
print('U', U, 'I', I, 'interactions', len(inter))
u_ids = u_inv.astype(np.int64)
i_ids = i_inv.astype(np.int64)
# best config confidence
alpha = 15.0
conf = (1.0 + alpha * ratings).astype(np.float32)
pref = (ratings > 0).astype(np.float32)
order = np.argsort(u_ids)
u_sorted = u_ids[order]; i_sorted = i_ids[order]; c_sorted = conf[order]; p_sorted = pref[order]
user_ptr = np.searchsorted(u_sorted, np.arange(U+1))
order_i = np.argsort(i_ids)
i_sorted_i = i_ids[order_i]; u_sorted_i = u_ids[order_i]; c_sorted_i = conf[order_i]; p_sorted_i = pref[order_i]
item_ptr = np.searchsorted(i_sorted_i, np.arange(I+1))


In [ ]:

FACTORS = 64
REG = 0.1
ITERS = 10
CHUNK_U = 1024
CHUNK_I = 2048
device0 = torch.device('cuda:0')
device1 = torch.device('cuda:1')

def init_factors(n, f, seed=42):
    g = torch.Generator().manual_seed(seed)
    return (torch.rand(n, f, generator=g) * 0.01).numpy().astype(np.float32)

X = init_factors(U, FACTORS)
Y = init_factors(I, FACTORS)

def process_user_chunk(start_u, end_u, Ymat, device):
    Uc = end_u - start_u
    A = torch.zeros(Uc, FACTORS, FACTORS, device=device)
    b = torch.zeros(Uc, FACTORS, device=device)
    A += (torch.eye(FACTORS, device=device) * REG).unsqueeze(0)
    lo = int(user_ptr[start_u]); hi = int(user_ptr[end_u])
    bs = 2048
    for s in range(lo, hi, bs):
        e = min(s+bs, hi)
        local = torch.from_numpy(u_sorted[s:e] - start_u).to(device)
        ii = i_sorted[s:e]
        cc = torch.from_numpy(c_sorted[s:e]).to(device)
        pp = torch.from_numpy(p_sorted[s:e]).to(device)
        Yi = torch.from_numpy(Ymat[ii]).to(device)
        outer = torch.einsum('mi,mj->mij', Yi, Yi) * cc[:, None, None]
        A.index_add_(0, local, outer)
        b.index_add_(0, local, Yi * (cc * pp)[:, None])
    Xc = torch.linalg.solve(A, b.unsqueeze(2)).squeeze(2)
    return Xc.detach().cpu().numpy()

def update_users(X, Y):
    i = 0; ci = 0
    while i < U:
        j = min(i+CHUNK_U, U)
        dev = device0 if (ci % 2 == 0) else device1
        X[i:j] = process_user_chunk(i, j, Y, dev)
        i = j; ci += 1

def process_item_chunk(start_i, end_i, Xmat, device):
    Ic = end_i - start_i
    A = torch.zeros(Ic, FACTORS, FACTORS, device=device)
    b = torch.zeros(Ic, FACTORS, device=device)
    A += (torch.eye(FACTORS, device=device) * REG).unsqueeze(0)
    lo = int(item_ptr[start_i]); hi = int(item_ptr[end_i])
    bs = 2048
    for s in range(lo, hi, bs):
        e = min(s+bs, hi)
        local = torch.from_numpy(i_sorted_i[s:e] - start_i).to(device)
        uu = u_sorted_i[s:e]
        cc = torch.from_numpy(c_sorted_i[s:e]).to(device)
        pp = torch.from_numpy(p_sorted_i[s:e]).to(device)
        Xu = torch.from_numpy(Xmat[uu]).to(device)
        outer = torch.einsum('mi,mj->mij', Xu, Xu) * cc[:, None, None]
        A.index_add_(0, local, outer)
        b.index_add_(0, local, Xu * (cc * pp)[:, None])
    Yc = torch.linalg.solve(A, b.unsqueeze(2)).squeeze(2)
    return Yc.detach().cpu().numpy()

def update_items(Y, X):
    i = 0; ci = 0
    while i < I:
        j = min(i+CHUNK_I, I)
        dev = device0 if (ci % 2 == 0) else device1
        Y[i:j] = process_item_chunk(i, j, X, dev)
        i = j; ci += 1

t0_all = time.time()
for it in range(ITERS):
    t0 = time.time()
    update_users(X, Y)
    update_items(Y, X)
    print(f'iter {it+1}/{ITERS} done in {time.time()-t0:.1f}s', flush=True)
print('TOTAL TRAIN TIME', time.time()-t0_all)


In [ ]:

out = Path('/kaggle/working')
np.save(out/'als_best_user_factors.npy', X)
np.save(out/'als_best_item_factors.npy', Y)
np.save(out/'als_best_user_ids.npy', u_uniq)
np.save(out/'als_best_item_ids.npy', i_uniq)
import json
meta = {'U': int(U), 'I': int(I), 'factors': FACTORS, 'iters': ITERS, 'alpha': alpha, 'reg': REG, 'interactions': int(len(inter))}
(out/'als_best_meta.json').write_text(json.dumps(meta))
print(json.dumps(meta, indent=2))


In [ ]:

# Evaluate RMSE/MAE on full test set
import math
u_all = inter['u'].to_numpy() if 'u' in inter else None
# map again (already have u_ids? we stored u_inv)
# Reuse contiguous arrays
u_idx = u_inv  # length len(inter)
i_idx = i_inv
r_all = ratings
# chunked prediction
chunk = 1_000_000
se = 0.0; ae = 0.0; n = 0
for s in range(0, len(inter), chunk):
    e = min(s+chunk, len(inter))
    uu = u_idx[s:e]; ii = i_idx[s:e]; rr = r_all[s:e]
    pred = np.sum(X[uu] * Y[ii], axis=1)
    se += float(np.sum((pred - rr)**2))
    ae += float(np.sum(np.abs(pred - rr)))
    n += (e - s)
rmse = math.sqrt(se/n); mae = ae/n
print(f'FULL TEST RMSE: {rmse:.4f} MAE: {mae:.4f} (n={n})')


In [ ]:

# Sampled ranking eval (5000 users)
rng = np.random.default_rng(7)
N_USERS = 5000
users_with_gt = np.unique(u_inv)
chosen = rng.choice(users_with_gt, size=min(N_USERS, len(users_with_gt)), replace=False)
item_pop = pd.Series(i_inv).value_counts()
all_items = np.arange(I)
hr10=0; ndcg10=0.0; mrr=0.0; cnt=0
for u in chosen:
    gt_rows = np.where(u_inv == u)[0]
    if len(gt_rows)==0: continue
    gt_item = int(i_inv[gt_rows[0]])
    negs = rng.choice(all_items, size=99, replace=False)
    negs = negs[negs != gt_item]
    cands = np.concatenate([[gt_item], negs])
    scores = X[u] @ Y[cands].T
    order = np.argsort(-scores)
    ranked = cands[order]
    pos = np.where(ranked == gt_item)[0]
    pos = int(pos[0]) if len(pos) else None
    if pos is not None and pos < 10:
        hr10 += 1
        ndcg10 += 1.0/np.log2(pos+2)
        mrr += 1.0/(pos+1)
    cnt += 1
print(f'Evaluated users: {cnt}')
print(f'ALS HR@10={hr10/cnt:.4f} NDCG@10={ndcg10/cnt:.4f} MRR={mrr/cnt:.4f}')
metrics = {
  'rmse_full': rmse,
  'mae_full': mae,
  'n_test': int(n),
  'hr10': hr10/cnt,
  'ndcg10': ndcg10/cnt,
  'mrr': mrr/cnt,
  'n_eval_users': cnt,
  'factors': FACTORS,
  'iters': ITERS,
  'alpha': alpha,
}
Path('/kaggle/working/als_best_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


In [ ]:
!ls -lh /kaggle/working